# Plot Learning Curves

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import glob

%load_ext autoreload
%autoreload 2

In [ ]:
ENV_NAMES = (
    # "CartPoleContinuous-v1",
    # "Pendulum-v1",
    "MountainCarContinuous-v0",
    # "LunarLanderContinuous-v3",
)
INPUT_DIR = "../results/test_rb_compositions-" + "selected"

In [ ]:
ENV_NAME = "MountainCarContinuous-v0"

PROPORTIONS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
SEEDS = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110]  # [0, 1, 2, 4]
EXPLOIT_REWARD = 61


def load_env_result(env_name, exploit_reward, proportions, seeds):
    d_ref = torch.load(
        f"../scripts/results-{env_name}-{exploit_reward}/result-{env_name}-{exploit_reward}-{proportions[0]}-{seeds[0]}.pt",
        weights_only=False,
    )
    res = {
        "performances": None,
        "buffer_size": d_ref["buffer_size"],
        "rb_composition": proportions,
        "eval_interval": d_ref["eval_interval"],
        "cfg": d_ref["cfg"],
        "seeds": seeds,
        "type": d_ref["type"],
    }
    res["performances"] = [
        np.stack(
            [
                torch.load(
                    f"../scripts/results-{env_name}-{exploit_reward}/result-MountainCarContinuous-v0-{exploit_reward}-{proportion}-{seed}.pt",
                    weights_only=False,
                )["performances"]
                for seed in SEEDS
            ],
            -1,
        )
        for proportion in proportions
    ]
    return res


mcc_61 = load_env_result("MountainCarContinuous-v0", EXPLOIT_REWARD, PROPORTIONS, SEEDS)

In [ ]:
# for i in range(len(SEEDS)):
#     plt.plot(mcc_35["performances"][2, :, :, i].mean(1))
# plt.show()

In [ ]:
all_performances = {}
for rb_composition_type in ("uniform_proportions", "noise_levels"):
    all_performances[rb_composition_type] = {}
    for env_name in ENV_NAMES:
        all_performances[rb_composition_type][env_name] = {}
        print(f"{env_name}:")
        for intermediate_path in glob.glob(
            f"{INPUT_DIR}/{rb_composition_type}-{env_name}-scoring-*"
        ):
            exploit_performance = float(intermediate_path.split("-scoring-")[1])
            print(f"    {exploit_performance}")
            all_performances[rb_composition_type][env_name][exploit_performance] = (
                torch.load(intermediate_path, weights_only=False)
            )

In [ ]:
all_performances["uniform_proportions"]["MountainCarContinuous-v0"][61] = mcc_61

In [ ]:
# print(all_performances["uniform_proportions"]["MountainCarContinuous-v0"][35])
# all_performances["uniform_proportions"]["MountainCarContinuous-v0"][35]["performances"][0].shape

In [ ]:
# print(all_performances["uniform_proportions"]["CartPoleContinuous-v1"][500])
# all_performances["uniform_proportions"]["CartPoleContinuous-v1"][500]["performances"][0].shape

In [ ]:
def smooth(x, window, mode="gaussian"):
    if window % 2 == 0:
        raise ValueError("Window size must be odd")

    radius = window // 2

    if mode == "gaussian":
        sigma = (window - 1) / 6  # key conversion

        t = np.arange(-radius, radius + 1)
        kernel = np.exp(-(t**2) / (2 * sigma**2))
        kernel /= kernel.sum()

        return np.convolve(x, kernel, mode="valid")

    elif mode == "max":
        return np.array([np.max(x[i : i + window]) for i in range(len(x) - window + 1)])

    else:
        raise ValueError("Unknown mode")

In [ ]:
import colorsys


def generate_colors(n, s=0.7, v=0.9):
    colors = []
    for i in range(n):
        h = i / n  # evenly spaced hue
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        colors.append(
            "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))
        )
    return colors

In [ ]:
# colors = ["#238b45", "#1d91c0", "#225ea8", "#253494", "#54278f", "#f03b20"] * 20
colors = generate_colors(11)
SMOOTH = True
SMOOTH_WINDOW = 11  # 11
SMOOTH_MODE = ("gaussian", "max")[0]
PLOT_ALL_CURVES = True
STEP_TO_TAKE = 15_000  # None  # 15_000
LAST_STEPS = 1000  # 500
PLOT_MARGIN = 0.05

last_performances = {}
std_last_performances = {}

for rb_composition_type in ("uniform_proportions", "noise_levels"):
    print("===========", rb_composition_type.upper(), "=============")
    for env_name in ENV_NAMES:
        if not all_performances[rb_composition_type][env_name]:
            continue

        last_performances[env_name] = {}
        std_last_performances[env_name] = {}

        # Learning curves ==========================
        if len(all_performances[rb_composition_type][env_name]) == 1:
            ncols = 1
        else:
            ncols = 2
        fig, axes = plt.subplots(
            int(np.ceil(len(all_performances[rb_composition_type][env_name]) / ncols)),
            ncols,
            figsize=(10, 10),
        )
        if ncols == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        for i_reward, reward in enumerate(
            sorted(all_performances[rb_composition_type][env_name])
        ):
            # NOTE: assumed to be the same
            rb_composition = all_performances[rb_composition_type][env_name][reward][
                "rb_composition"
            ]
            eval_interval = all_performances[rb_composition_type][env_name][reward][
                "eval_interval"
            ]

            print(rb_composition_type)
            last_performances[env_name][reward] = np.zeros(len(rb_composition))
            std_last_performances[env_name][reward] = np.zeros(len(rb_composition))
            # List by proportion of arrays EVAL_NUM x ENV x SEED
            performances = all_performances[rb_composition_type][env_name][reward][
                "performances"
            ]

            min_perf, max_perf = np.inf, -np.inf

            for i_prop, proportion in enumerate(rb_composition):
                nb_evals, nb_envs, nb_seeds = performances[0].shape
                if SMOOTH:
                    learning_curves = np.empty(
                        (nb_evals - SMOOTH_WINDOW + 1, nb_envs, nb_seeds)
                    )
                else:
                    learning_curves = np.empty((nb_evals, nb_envs, nb_seeds))
                for i_env in range(performances[0].shape[1]):
                    for i_seed in range(performances[0].shape[2]):
                        learning_curve = performances[i_prop][:, i_env, i_seed]
                        if SMOOTH:
                            learning_curve = smooth(
                                learning_curve, window=SMOOTH_WINDOW, mode=SMOOTH_MODE
                            )
                        learning_curves[:, i_env, i_seed] = learning_curve
                        label = proportion if i_seed == 0 and i_env == 0 else None
                        if PLOT_ALL_CURVES:
                            axes[i_reward].plot(
                                (np.arange(len(learning_curve)) + 1) * eval_interval,
                                learning_curve,
                                color=colors[i_prop],
                                alpha=0.03,
                            )  # label = label
                # NOTE: Choose aggregation: first on ENV and then on SEED dimension
                aggregated_learning_curves = learning_curves.mean(1).mean(1)
                std_learning_curves = learning_curves.mean(1).std(1)
                if STEP_TO_TAKE is not None:
                    step_slice = slice(
                        (STEP_TO_TAKE - LAST_STEPS) // eval_interval,
                        STEP_TO_TAKE // eval_interval,
                    )
                else:
                    step_slice = slice(-LAST_STEPS // eval_interval, None)

                last_performances[env_name][reward][i_prop] = (
                    aggregated_learning_curves[step_slice].mean()
                )
                std_last_performances[env_name][reward][i_prop] = std_learning_curves[
                    step_slice
                ].mean()  # TODO: better std pooling

                axes[i_reward].plot(
                    (np.arange(len(learning_curves)) + 1) * eval_interval,
                    aggregated_learning_curves,
                    color=colors[i_prop],
                    label=proportion,
                    alpha=1,
                )

                min_perf = np.minimum(learning_curves.min(), min_perf)
                max_perf = np.maximum(learning_curves.max(), max_perf)
                y_lim_range = np.abs(max_perf - min_perf)
                axes[i_reward].set_ylim(
                    [
                        min_perf - PLOT_MARGIN * y_lim_range,
                        max_perf + PLOT_MARGIN * y_lim_range,
                    ]
                )
            axes[i_reward].set_title(f"Exploit reward: {reward}")
            axes[i_reward].set_ylabel("evaluation performance")
            axes[i_reward].set_xlabel("nb steps")
        for ax in axes[i_reward + 1 :]:
            fig.delaxes(ax)
        axes[0].legend(
            title="Uniform\nproportion"
            if rb_composition_type == "uniform_proportions"
            else "Action\nnoise",
            loc="upper left",
        )
        fig.suptitle(
            f"{env_name}" + f"\n{SMOOTH_MODE} smooth (w={SMOOTH_WINDOW})"
            if SMOOTH
            else ""
        )
        plt.tight_layout()
        plt.show()

        # Performance ~ RB composition =====================================

        plt.figure(figsize=(10, 10))
        for i_reward, reward in enumerate(
            sorted(all_performances[rb_composition_type][env_name])
        ):
            performances = all_performances[rb_composition_type][env_name][reward]
            means = last_performances[env_name][reward]
            stds = std_last_performances[env_name][reward]
            plt.plot(rb_composition, means, label=reward)
            plt.fill_between(
                rb_composition,
                means - stds,
                means + stds,
                alpha=0.1,
            )

        plt.title(
            f"{env_name}\nperformance in {step_slice.start * eval_interval, step_slice.stop * eval_interval if step_slice.stop is not None else 'max'} steps"
        )
        plt.xlabel(
            "% of uniform exploration in replay buffer"
            if rb_composition_type == "uniform_proportions"
            else "truncated gaussian noise on policy actions"
        )
        plt.ylabel("evaluation performance (mean $\\pm$ std)")
        plt.legend(title="Exploit reward")
        plt.tight_layout()
        plt.show()
